This notebook plots the results of notebook 03: how the linear probe trained, how each method's setting was chosen, and how well each of the 6 models (2 methods × 3 feature sets) predicts crops on the held-out test blocks, overall, by country and by crop. Three scores are shown: accuracy (share of fields labelled correctly), mean recall (share of each crop's fields found, averaged over crops) and macro F1 (balances recall with precision, averaged over crops).

In [ ]:
import numpy as np
import pandas as pd
import ultraplot as uplt
from pathlib import Path
from sklearn.metrics import f1_score,recall_score

In [ ]:
INTERIMDIR  = Path('../data/interim')
PROCDIR     = Path('../data/processed')
FIGDIR      = Path('../figs')
COLLECTIONS = pd.read_csv(INTERIMDIR/'collections.csv')
NAMES       = dict(zip(COLLECTIONS.collection.str.split('/').str[1],COLLECTIONS.name))
METHODS     = {'probe':'Linear probe','knn':'kNN'}
COLORS      = {'aef':'#2a78d6','olmoearth':'#eb6834','both':'#1baf7a'}
METRICS     = {'accuracy':'Accuracy','meanrecall':'Mean recall','macrof1':'Macro F1'}
TOPCROPS    = 25

FIGDIR.mkdir(parents=True,exist_ok=True)

In [ ]:
predictions = pd.read_parquet(INTERIMDIR/'results_predictions.parquet').assign(country=lambda table:table.country.map(NAMES))
losses      = pd.read_csv(PROCDIR/'results_probelosses.csv')
probesweep  = pd.read_csv(PROCDIR/'results_probesweep.csv')
knnsweep    = pd.read_csv(PROCDIR/'results_knnsweep.csv')
models      = [(method,featureset) for method in METHODS for featureset in COLORS]

In [ ]:
def score(group,method,featureset):
    truth,pred = group.crop,group[f'{method}_{featureset}']
    crops      = truth.unique()
    return {'accuracy':(truth==pred).mean(),
            'meanrecall':recall_score(truth,pred,labels=crops,average='macro',zero_division=0),
            'macrof1':f1_score(truth,pred,labels=crops,average='macro',zero_division=0)}

scores = pd.DataFrame([{'country':country,'method':method,'featureset':featureset,**score(group,method,featureset)}
                       for country,group in [('All countries',predictions)]+list(predictions.groupby('country')) for method,featureset in models])
scores.to_csv(PROCDIR/'results_scores.csv',index=False)

## Training the linear probe

Left: training (dashed) and validation (solid) loss of the final probe for each feature set, by optimiser iteration. Right: validation loss during the penalty sweep, one line per penalty. Validation loss rising while training loss keeps falling means the probe is overfitting; each fit keeps the weights from its lowest validation loss.

In [ ]:
fig,axs = uplt.subplots(ncols=2,refwidth=3.5,share=False)
for featureset,color in COLORS.items():
    curve = losses.query("stage=='final' and featureset==@featureset")
    axs[0].plot(curve.iterations.to_numpy(),curve.trainloss.to_numpy(),color=color,linestyle='--')
    axs[0].plot(curve.iterations.to_numpy(),curve.valloss.to_numpy(),color=color,label=featureset)
for penalty,curve in losses.query("stage=='sweep'").groupby('penalty'):
    axs[1].plot(curve.iterations.to_numpy(),curve.valloss.to_numpy(),label=f'{penalty:,}')
axs[0].format(title='Final probes (dashed: training, solid: validation)',xlabel='Iterations',ylabel='Loss')
axs[1].format(title='Penalty sweep, both feature sets (validation)',xlabel='Iterations',ylabel='Loss')
axs[0].legend(loc='ur',ncols=1,frame=False)
axs[1].legend(loc='ur',ncols=1,frame=False,title='Penalty')
fig.savefig(FIGDIR/'04_probe_losses.png',dpi=200)

## Choosing each method's setting

Validation macro F1 for each penalty (linear probe) and each k (kNN). The best value on the validation fields is the one used on the test fields.

In [ ]:
fig,axs = uplt.subplots(ncols=2,refwidth=3,share=False)
axs[0].plot(probesweep.penalty,probesweep.valmacrof1,color=COLORS['both'],marker='o',label='both')
for featureset,color in COLORS.items():
    sweep = knnsweep.query('featureset==@featureset')
    axs[1].plot(sweep.k,sweep.valmacrof1,color=color,marker='o',label=featureset)
axs[0].format(title='Linear probe',xlabel='Penalty',ylabel='Validation macro F1',xscale='log')
axs[1].format(title='kNN',xlabel='k (neighbours)',ylabel='Validation macro F1')
axs[1].legend(loc='ur',ncols=1,frame=False)
fig.savefig(FIGDIR/'04_validation_sweeps.png',dpi=200)

## Scores on the test blocks

In [ ]:
overall = scores[scores.country=='All countries']

fig,axs = uplt.subplots(ncols=3,refwidth=2.2,refheight=1.2,sharey=True,sharex=False)
for ax,(metric,label) in zip(axs,METRICS.items()):
    for featureset,color in COLORS.items():
        subset = overall[overall.featureset==featureset]
        ax.scatter(subset[metric],subset.method.map(METHODS),color=color,label=featureset,s=40)
    ax.format(title=label,xlabel='',ylabel='',xlim=(0,1),ygrid=True)
fig.legend(axs[0].get_legend_handles_labels()[0][:3],list(COLORS),loc='b',ncols=3,frame=False)
fig.format(suptitle='All countries')
fig.savefig(FIGDIR/'04_scores_overall.png',dpi=200)

In [ ]:
bycountry = scores[scores.country!='All countries']
order     = bycountry.query("method=='probe' and featureset=='both'").sort_values('macrof1').country.tolist()

fig,axs = uplt.subplots(nrows=2,ncols=3,refwidth=2.2,refheight=0.22*len(order),sharey=True,sharex=False)
for row,method in enumerate(METHODS):
    for col,(metric,label) in enumerate(METRICS.items()):
        ax = axs[row,col]
        for featureset,color in COLORS.items():
            subset = bycountry[(bycountry.method==method)&(bycountry.featureset==featureset)].set_index('country').loc[order]
            ax.scatter(subset[metric],subset.index,color=color,label=featureset,s=20)
        ax.format(title=f'{METHODS[method]}: {label}',xlabel='',ylabel='',xlim=(0,1),ygrid=True)
fig.legend(axs[0].get_legend_handles_labels()[0][:3],list(COLORS),loc='b',ncols=3,frame=False)
fig.savefig(FIGDIR/'04_scores_by_country.png',dpi=200)

In [ ]:
top    = predictions.crop.value_counts().head(TOPCROPS).index[::-1]
recall = pd.DataFrame({(method,featureset):recall_score(predictions.crop,predictions[f'{method}_{featureset}'],labels=top,average=None,zero_division=0)
                       for method,featureset in models},index=top)

fig,axs = uplt.subplots(ncols=2,refwidth=2.5,refheight=0.2*TOPCROPS,sharey=True)
for ax,method in zip(axs,METHODS):
    for featureset,color in COLORS.items():
        ax.scatter(recall[(method,featureset)],top,color=color,label=featureset,s=20)
    ax.format(title=METHODS[method],xlabel='Recall',ylabel='',xlim=(0,1),ygrid=True,yticklabelsize=7)
fig.legend(axs[0].get_legend_handles_labels()[0][:3],list(COLORS),loc='b',ncols=3,frame=False)
fig.format(suptitle=f'Recall for the {TOPCROPS} most common crops (all countries)')
fig.savefig(FIGDIR/'04_recall_by_crop.png',dpi=200)

In [ ]:
top = predictions.crop.value_counts().head(TOPCROPS).index

fig,axs = uplt.subplots(nrows=2,ncols=3,refwidth=3.2,share=False)
for row,method in enumerate(METHODS):
    for col,featureset in enumerate(COLORS):
        counts = pd.crosstab(predictions.crop,predictions[f'{method}_{featureset}'])
        matrix = counts.reindex(index=top,columns=top,fill_value=0).div(counts.loc[top].sum(axis=1),axis=0)
        axs[row,col].imshow(matrix.to_numpy(),cmap='Blues',vmin=0,vmax=1)
        axs[row,col].format(title=f'{METHODS[method]}: {featureset}',xlabel='Predicted crop',ylabel='True crop',grid=False,
                            xticks=range(len(top)),yticks=range(len(top)),xticklabels=list(top) if row==1 else [],
                            yticklabels=list(top) if col==0 else [],xrotation=90,ticklabelsize=5,xtickminor=False,ytickminor=False)
fig.format(suptitle=f'Share of each true crop predicted as each crop ({TOPCROPS} most common crops, all countries)')
fig.savefig(FIGDIR/'04_confusion.png',dpi=200)